---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** __장다예__ &nbsp; **학번:** ____20251281____ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. cancer.xlsx의 Data 시트 불러오기
df = pd.read_excel('cancer.xlsx', sheet_name='Data')
df.head()

In [ ]:
# 데이터 내 존재하는 종(Species)의 분포 확인
df['Species'].value_counts()

In [ ]:
# 2. 마우스(Mouse) 실험 결과이면서 단일 화학물질(Chemical)만 선택하여 필터링
df_m = df[(df['Species'] == 'Mouse') & (df['Mixture'] == 'Chemical')].copy()
df_m

In [ ]:
# 마우스 데이터 내에 존재하는 엔드포인트(Endpoint) 종류 및 빈도 확인
df_m['Endpoint'].value_counts()

In [ ]:
# 3. 발암성 평가 결과에 해당하는 'Level of evidence of carcinogenic activity'를 가진 데이터 추출
df_evi = df_m[df_m['Endpoint'] == 'Level of evidence of carcinogenic activity'].copy()

# 머신러닝 학습에 노이즈를 유발할 수 있는 불확실한 결과(Equivocal, Inadequate, Not tested 등) 제거
valid_responses = ['Clear evidence', 'Some evidence', 'No evidence']
df_evi = df_evi[df_evi['Response'].isin(valid_responses)].copy()
df_evi.shape

In [ ]:
# 필터링된 최종 발암성 응답 값들의 분포 확인
df_evi['Response'].value_counts()

In [ ]:
# 응답(Response) 값 분포 시각화
import matplotlib.pyplot as plt
evi_value = df_evi['Response'].value_counts()
plt.bar(evi_value.index, evi_value)
plt.xticks(rotation=45)
plt.title('Distribution of Carcinogenicity Response (Mouse)')
plt.show()

In [ ]:
# 데이터 칼럼 확인
df_evi.columns

In [ ]:
# 4. 타겟 변수 이진 라벨링 (Active/발암성 있음 = 1, Inactive/발암성 없음 = 0)
df_evi['label'] = 0
df_evi.loc[df_evi['Response'].isin(['Clear evidence', 'Some evidence']),'label'] = 1

df_evi['label'].value_counts()

In [ ]:
label_count = df_evi.groupby('SMILES')['label'].nunique()
label_count

In [ ]:
valid_smiles = label_count[label_count == 1].index
df_evi = df_evi[df_evi['SMILES'].isin(valid_smiles)]
df_evi = df_evi.drop_duplicates(subset='SMILES')
df_evi = df_evi.drop(columns=['Sex'])
print(df_evi.shape)

In [ ]:
df_evi.to_csv('cancer_evi.csv')

In [ ]:
from rdkit import Chem

df_evi['mol'] = df_evi['SMILES'].apply(Chem.MolFromSmiles)

df_ml.columns

In [ ]:
import pandas as pd

descrs = []

for mol in df_evi['mol']:
    calc = Descriptors.CalcMolDescriptors(mol)
    descrs.append(calc)

df_descrs = pd.DataFrame(descrs)

df_ml = pd.concat(
    [df_evi[['SMILES', 'label']].reset_index(drop=True),
     df_descrs.reset_index(drop=True)],
    axis=1
)

print(df_ml.shape)

df_ml

In [ ]:
X = df_ml.drop(columns=['SMILES', 'label', 'mol'], errors='ignore')
y = df_ml['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})

importance.sort_values(
    by='importance',
    ascending=False
).head(20)